In [ ]:
import pygame
import cv2
import numpy as np
import random
import math
import os

pygame.init()

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
WIDTH, HEIGHT = 1100, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Balloon Pop Game")

clock = pygame.time.Clock()
font = pygame.font.SysFont("Arial", 32, bold=True)

BALLOON_PATH = os.path.join(script_dir, "Resources", "Balloon.png")

balloon_image = pygame.image.load(BALLOON_PATH).convert_alpha()
balloon_image = pygame.transform.smoothscale(balloon_image, (85, 115))

def Webcam(cameras : int):
    if type(cameras) == int and cameras:
        return cv2.VideoCapture(cameras)
    
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Access Webcam")

balloons = []
score = 0

SPAWN_INTERVAL = 900
last_spawn = pygame.time.get_ticks()

def spawn_balloon():
    balloons.append({"x": random.randint(60, WIDTH - 60),"y": HEIGHT + 100,"speed": random.uniform(1.5, 3.5)})


def detect_finger(frame):
    frame = cv2.flip(frame, 1)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60])
    upper_skin = np.array([25, 180, 255])
    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest_contour) < 2500:
        return None
    topmost = tuple(largest_contour[largest_contour[:, :, 1].argmin()][0])

    camera_height, camera_width = frame.shape[:2]
    game_x = int(topmost[0] / camera_width * WIDTH)
    game_y = int(topmost[1] / camera_height * HEIGHT)

    return game_x, game_y

running = True

while running:

    clock.tick(60)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    success, camera_frame = camera.read()

    fingertip = None

    if success:
        fingertip = detect_finger(camera_frame)

    current_time = pygame.time.get_ticks()

    if current_time - last_spawn >= SPAWN_INTERVAL:
        spawn_balloon()
        last_spawn = current_time

    screen.fill((135, 206, 235))

    for balloon in balloons[:]:
        balloon["y"] -= balloon["speed"]
        balloon_rect = balloon_image.get_rect(center=(int(balloon["x"]), int(balloon["y"])))

        screen.blit(balloon_image, balloon_rect)

        if fingertip is not None:
            distance = math.dist(fingertip,(balloon["x"], balloon["y"]))

            if distance < 45:
                balloons.remove(balloon)
                score += 1
                continue

        if balloon["y"] < -100:
            balloons.remove(balloon)

    score_rect = pygame.Rect(WIDTH - 220, 20, 190, 65)

    pygame.draw.rect(screen,(255, 255, 255),score_rect,border_radius=12)
    score_text = font.render(f"Score : {score}",True,(30, 30, 30))
    screen.blit(score_text, (WIDTH - 205, 35))
    pygame.display.flip()


camera.release()
pygame.quit()
cv2.destroyAllWindows()